# Week 4 - Quantisation sweep (Log-Triage SLM)

Makes the missing quantisation levels of the **fine-tuned** SmolLM3-3B from the merged model saved by the
Week 3 notebook (Version 2 output). Q8_0 already exists and is **reused, not rebuilt**.

| Level  | Made by | From |
|--------|---------|------|
| F16    | `convert_hf_to_gguf.py --outtype f16` | merged model (bf16) |
| Q5_K_M | `llama-quantize` | the F16 GGUF |
| Q4_K_M | `llama-quantize` | the F16 GGUF |
| Q3_K_M | `llama-quantize` | the F16 GGUF |

Same pinned llama.cpp commit as Week 3. No importance matrix (plain quantisation).

**Before running:** Accelerator **None** (CPU is enough) · Internet **On** · Input: the Week 3 notebook's
**Version 2** output · paste your local Q8_0 hash into cell 1.

In [ ]:
# 1) Settings. The ONLY line you edit is LOCAL_Q8_SHA256.
#    Get it on your PC with:  Get-FileHash models\smollm3-logtriage-q8_0.gguf -Algorithm SHA256
LOCAL_Q8_SHA256 = "PASTE_YOUR_HASH_HERE"

LLAMA_CPP_COMMIT = "b23efaa2ef147f547ee75cbf0c621d61904de80e"   # same converter as Week 3 - never change
INPUT_DIR = "/kaggle/input"
OUT       = "/kaggle/working"          # only the 4 new GGUFs + week4_quant_info.json are saved here
WORK      = "/tmp/w4"                  # llama.cpp source + build + temporary files (not saved)
Q8_NAME   = "smollm3-logtriage-q8_0.gguf"
F16_NAME  = "smollm3-logtriage-f16.gguf"
K_LEVELS  = ["Q5_K_M", "Q4_K_M", "Q3_K_M"]
def gguf_name(level):
    return f"smollm3-logtriage-{level.lower()}.gguf"

import os, sys, json, time, glob, shutil, hashlib, subprocess, platform
import numpy as np
os.makedirs(WORK, exist_ok=True)
T0 = time.time()
TIMES = {}

def run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    if r.returncode != 0:
        raise SystemExit("Command failed - paste the output above to Claude.")

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(16 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("Settings OK. CPU cores:", os.cpu_count())

In [ ]:
# 2) Find the Week 3 output and prove it is the exact version you scored on your PC.
t = time.time()
hits = [os.path.join(d, Q8_NAME) for d, _, files in os.walk(INPUT_DIR) if Q8_NAME in files]
if len(hits) != 1:
    raise SystemExit(f"Expected exactly 1 {Q8_NAME} under {INPUT_DIR}, found {len(hits)}: {hits}\n"
                     "Add the Week 3 notebook's Version 2 output as input (and only that version).")
Q8 = hits[0]
W3_DIR = os.path.dirname(Q8)
MERGED = os.path.join(W3_DIR, "merged")
SHARDS = sorted(glob.glob(os.path.join(MERGED, "*.safetensors")))
if not (os.path.exists(os.path.join(MERGED, "config.json")) and SHARDS):
    raise SystemExit(f"No merged model next to the Q8_0 file. Contents of {W3_DIR}: {sorted(os.listdir(W3_DIR))}")
print("Week 3 output :", W3_DIR)
print("Merged model  :", sorted(os.listdir(MERGED)))

if LOCAL_Q8_SHA256.startswith("PASTE"):
    raise SystemExit("Paste your local Q8_0 hash into LOCAL_Q8_SHA256 in cell 1 first.")
Q8_SHA = sha256(Q8)
print("Q8_0 on Kaggle:", Q8_SHA)
print("Q8_0 on PC    :", LOCAL_Q8_SHA256.strip().lower())
if Q8_SHA != LOCAL_Q8_SHA256.strip().lower():
    raise SystemExit("MISMATCH: this input is not the version you scored. Attach the Week 3 Version 2 output.")
print("MATCH: this merged model is the one behind the Q8_0 you scored.")

w3_info = os.path.join(W3_DIR, "run_info.json")
if os.path.exists(w3_info):
    info = json.load(open(w3_info))
    print("Week 3 run_info: best checkpoint =", info.get("best_checkpoint"),
          "| llama.cpp =", info.get("versions", {}).get("llama_cpp_commit"))
    assert info.get("versions", {}).get("llama_cpp_commit") == LLAMA_CPP_COMMIT, "Week 3 used a different llama.cpp commit - paste this to Claude"

for path, need_gb in [(OUT, 14), (WORK, 6)]:
    free = shutil.disk_usage(path).free / 1e9
    print(f"Free disk at {path}: {free:.1f} GB (need {need_gb})")
    if free < need_gb:
        raise SystemExit(f"Not enough disk at {path} - paste this to Claude.")
TIMES["find_and_verify_input_s"] = round(time.time() - t)

In [ ]:
# 3) Get llama.cpp at the pinned commit and build ONLY the llama-quantize tool (CPU, ~3-6 min).
#    Red "ERROR: pip's dependency resolver ..." lines about OTHER packages are normal on Kaggle - ignore them.
%pip install -q "transformers==5.17.0" "sentencepiece==0.2.2"
t = time.time()
LLAMA = f"{WORK}/llama.cpp"
if not os.path.exists(f"{LLAMA}/convert_hf_to_gguf.py"):
    run(f"git init -q {LLAMA} && cd {LLAMA} && git remote add origin https://github.com/ggml-org/llama.cpp "
        f"&& git fetch -q --depth 1 origin {LLAMA_CPP_COMMIT} && git checkout -q FETCH_HEAD")
run(f"cd {LLAMA} && git rev-parse HEAD")
assert subprocess.check_output(["git", "-C", LLAMA, "rev-parse", "HEAD"], text=True).strip() == LLAMA_CPP_COMMIT

if shutil.which("cmake") is None:
    run(f"{sys.executable} -m pip install -q cmake")
QUANTIZE = f"{LLAMA}/build/bin/llama-quantize"
if not os.path.exists(QUANTIZE):
    run(f"cd {LLAMA} && cmake -B build -DGGML_CUDA=OFF -DLLAMA_OPENSSL=OFF -DLLAMA_BUILD_TESTS=OFF "
        f"-DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_SERVER=OFF -DLLAMA_BUILD_APP=OFF")
    run(f"cd {LLAMA} && cmake --build build --target llama-quantize -j {os.cpu_count()}")
assert os.path.exists(QUANTIZE), "llama-quantize was not built - paste the output above to Claude"

sys.path.insert(0, f"{LLAMA}/gguf-py")
import gguf
import transformers
print("llama-quantize ready:", QUANTIZE, "| transformers", transformers.__version__)
TIMES["build_tools_s"] = round(time.time() - t)

In [ ]:
# 4) F16: convert the merged model to an F16 GGUF with the same converter as Week 3 (~3-6 min).
t = time.time()
F16 = f"{OUT}/{F16_NAME}"
run(f"{sys.executable} {LLAMA}/convert_hf_to_gguf.py {MERGED} --outtype f16 --outfile {F16}")
print(f"F16 written: {F16}  ({os.path.getsize(F16) / 1e9:.2f} GB)")
TIMES["convert_f16_s"] = round(time.time() - t)

In [ ]:
# 5) What did storing the bf16 weights as F16 change?  (~2-4 min)
#    bf16 and f16 are both 16-bit, but f16 covers a smaller range of sizes. Very large numbers would
#    become "infinity" (fatal) and extremely small ones round to zero or lose digits. We count both.
t = time.time()
F16_MAX = 65504.0
counts = dict(weights=0, overflow_to_inf=0, flushed_to_zero=0, small_values_rounded=0, already_not_finite=0)
max_abs = 0.0
not_bf16 = []
with np.errstate(over="ignore"):
    for shard in SHARDS:
        with open(shard, "rb") as f:
            n = int.from_bytes(f.read(8), "little")
            header = json.loads(f.read(n))
        raw_all = np.memmap(shard, dtype=np.uint8, mode="r")
        for name, meta in header.items():
            if name == "__metadata__":
                continue
            if meta["dtype"] != "BF16":
                not_bf16.append((name, meta["dtype"]))
                continue
            s, e = meta["data_offsets"]
            raw = raw_all[8 + n + s: 8 + n + e].view(np.uint16)
            for i in range(0, raw.size, 32_000_000):
                x = (raw[i:i + 32_000_000].astype(np.uint32) << 16).view(np.float32)   # exact bf16 -> f32
                h = x.astype(np.float16).astype(np.float32)                           # what the converter does
                fin = np.isfinite(x)
                counts["weights"] += x.size
                counts["already_not_finite"] += int((~fin).sum())
                counts["overflow_to_inf"] += int((fin & np.isinf(h)).sum())
                counts["flushed_to_zero"] += int(((h == 0) & (x != 0)).sum())
                counts["small_values_rounded"] += int(((h != x) & (h != 0) & np.isfinite(h)).sum())
                if fin.any():
                    max_abs = max(max_abs, float(np.abs(x[fin]).max()))
BF16_TO_F16 = dict(counts, max_abs_weight=max_abs, f16_max=F16_MAX, non_bf16_tensors=not_bf16)
for k, v in BF16_TO_F16.items():
    print(f"  {k:22s} {v}")
if counts["overflow_to_inf"] or counts["already_not_finite"]:
    raise SystemExit("Some weights became (or already were) infinity/NaN - paste this to Claude.")

# Direct check on the file itself: no infinity/NaN anywhere in the F16 GGUF.
bad = 0
for tensor in gguf.GGUFReader(F16).tensors:
    if tensor.tensor_type in (gguf.GGMLQuantizationType.F16, gguf.GGMLQuantizationType.F32):
        d = tensor.data.reshape(-1)
        for i in range(0, d.size, 32_000_000):
            bad += int((~np.isfinite(d[i:i + 32_000_000])).sum())
BF16_TO_F16["f16_gguf_non_finite"] = bad
if bad:
    raise SystemExit(f"F16 GGUF contains {bad} infinity/NaN values - paste this to Claude.")
pct = 100 * (counts["flushed_to_zero"] + counts["small_values_rounded"]) / counts["weights"]
print(f"F16 check passed: no infinity/NaN. {pct:.6f}% of weights were tiny values that F16 stores less exactly.")
TIMES["f16_check_s"] = round(time.time() - t)

In [ ]:
# 6) Q5_K_M, Q4_K_M, Q3_K_M: llama-quantize from the F16 GGUF (~1-3 min each).
t = time.time()
THREADS = os.cpu_count()
for level in K_LEVELS:
    run(f"{QUANTIZE} {F16} {OUT}/{gguf_name(level)} {level} {THREADS}")
    print(f"{level} written: {os.path.getsize(f'{OUT}/{gguf_name(level)}') / 1e9:.2f} GB")
TIMES["quantize_s"] = round(time.time() - t)

In [ ]:
# 7) Safety checks.
#  a) Every new GGUF has the same settings as the Week 3 Q8_0 (architecture, RoPE, tokenizer, BOS flag,
#     vocabulary, chat template). Only "general.*" keys (name, file type) may differ.
#  b) Same tensors (names + shapes) as the Q8_0.
#  c) Report which storage type each level actually uses (K-quant levels are a MIX of types).
t = time.time()
def gguf_meta(path):
    return {k: f.contents() for k, f in gguf.GGUFReader(path).fields.items() if not k.startswith("GGUF.")}
def tensor_info(path):
    return {tt.name: (tuple(int(x) for x in tt.shape), tt.tensor_type.name) for tt in gguf.GGUFReader(path).tensors}

Q8_META, Q8_TENSORS = gguf_meta(Q8), tensor_info(Q8)
EXPECTED_FTYPE = {"F16": gguf.LlamaFileType.MOSTLY_F16, "Q5_K_M": gguf.LlamaFileType.MOSTLY_Q5_K_M,
                  "Q4_K_M": gguf.LlamaFileType.MOSTLY_Q4_K_M, "Q3_K_M": gguf.LlamaFileType.MOSTLY_Q3_K_M,
                  "Q8_0": gguf.LlamaFileType.MOSTLY_Q8_0}
LEVEL_FILES = {"F16": F16, **{lv: f"{OUT}/{gguf_name(lv)}" for lv in K_LEVELS}}
TYPE_MIX = {}
for level, path in {"Q8_0": Q8, **LEVEL_FILES}.items():
    meta = Q8_META if path == Q8 else gguf_meta(path)
    tens = Q8_TENSORS if path == Q8 else tensor_info(path)
    diff = sorted(k for k in set(Q8_META) | set(meta)
                  if not k.startswith("general.") and Q8_META.get(k) != meta.get(k))
    if diff:
        raise SystemExit(f"{level}: settings differ from the Week 3 Q8_0: {diff} - paste this to Claude.")
    if {k: v[0] for k, v in tens.items()} != {k: v[0] for k, v in Q8_TENSORS.items()}:
        raise SystemExit(f"{level}: tensor names/shapes differ from the Week 3 Q8_0 - paste this to Claude.")
    if meta.get("general.file_type") != int(EXPECTED_FTYPE[level]):
        raise SystemExit(f"{level}: file type is {meta.get('general.file_type')}, expected {int(EXPECTED_FTYPE[level])}")
    mix = {}
    for _, ttype in tens.values():
        mix[ttype] = mix.get(ttype, 0) + 1
    TYPE_MIX[level] = dict(sorted(mix.items(), key=lambda kv: -kv[1]))
    print(f"{level:7s} OK  tensors={len(tens)}  types={TYPE_MIX[level]}")
print("All levels: same settings, same tokenizer/BOS flag, same tensors as the Week 3 Q8_0.")
print("add_bos_token in file:", Q8_META.get("tokenizer.ggml.add_bos_token"), "| pre-tokenizer:", Q8_META.get("tokenizer.ggml.pre"))
TIMES["checks_s"] = round(time.time() - t)

In [ ]:
# 8) Report detail: Week 3's Q8_0 was made directly from bf16 by the converter; the other levels go through
#    F16 + llama-quantize. Make a throwaway Q8_0 the Week 4 way and compare weights. NOT saved, NOT used.
t = time.time()
TMP_Q8 = f"{WORK}/q8_route_check.gguf"
run(f"{QUANTIZE} {F16} {TMP_Q8} Q8_0 {THREADS}")
a, b = gguf.GGUFReader(Q8), {tt.name: tt for tt in gguf.GGUFReader(TMP_Q8).tensors}
same_t, diff_t, diff_bytes, total_bytes = 0, [], 0, 0
for ta in a.tensors:
    tb = b[ta.name]
    da, db = np.asarray(ta.data).view(np.uint8).reshape(-1), np.asarray(tb.data).view(np.uint8).reshape(-1)
    total_bytes += da.size
    if ta.tensor_type == tb.tensor_type and da.size == db.size and np.array_equal(da, db):
        same_t += 1
    else:
        diff_t.append((ta.name, ta.tensor_type.name, tb.tensor_type.name))
        if da.size == db.size:
            diff_bytes += int((da != db).sum())
del a, b
os.remove(TMP_Q8)
Q8_ROUTE = dict(identical_tensors=same_t, different_tensors=len(diff_t), different_tensor_list=diff_t[:20],
                differing_bytes=diff_bytes, total_bytes=total_bytes)
print(f"Q8_0 two routes: {same_t} tensors identical, {len(diff_t)} different, "
      f"{diff_bytes} of {total_bytes} bytes differ ({100 * diff_bytes / total_bytes:.6f}%).")
TIMES["q8_route_check_s"] = round(time.time() - t)

In [ ]:
# 9) Record sizes + hashes for the report, then list what to download.
t = time.time()
levels = {"F16": F16, "Q8_0 (Week 3, reused)": Q8, **{lv: f"{OUT}/{gguf_name(lv)}" for lv in K_LEVELS}}
files = {}
for level, path in levels.items():
    files[level] = dict(file=os.path.basename(path), bytes=os.path.getsize(path),
                        gb=round(os.path.getsize(path) / 1e9, 3),
                        sha256=Q8_SHA if path == Q8 else sha256(path),
                        type_mix=TYPE_MIX[level.split(" ")[0]])
TIMES["hash_s"] = round(time.time() - t)
TIMES["total_min"] = round((time.time() - T0) / 60, 1)
QUANT_INFO = dict(
    llama_cpp_commit=LLAMA_CPP_COMMIT,
    source="Week 3 Version 2 merged model (bf16); Q8_0 hash verified against the scored local copy",
    method={"F16": "convert_hf_to_gguf.py --outtype f16 from merged bf16",
            "K-quants": "llama-quantize from the F16 GGUF, no importance matrix",
            "Q8_0": "Week 3: convert_hf_to_gguf.py --outtype q8_0 from merged bf16 (reused, not rebuilt)"},
    files=files, bf16_to_f16=BF16_TO_F16, q8_route_check=Q8_ROUTE, times=TIMES,
    versions=dict(python=platform.python_version(), numpy=np.__version__, transformers=transformers.__version__,
                  cpu_cores=os.cpu_count()))
json.dump(QUANT_INFO, open(f"{OUT}/week4_quant_info.json", "w"), indent=2)

print(f"{'Level':24s} {'File':36s} {'Bytes':>15s}")
for level, f in files.items():
    print(f"{level:24s} {f['file']:36s} {f['bytes']:>15,d}")
print("\nDONE in", TIMES["total_min"], "min. Download from the Output tab into C:\\FineTuning\\models\\ :")
for lv in ["F16"] + K_LEVELS:
    print("  ", files[lv]["file"])
print("and week4_quant_info.json into C:\\FineTuning\\results\\")
print("Saved here:", sorted(os.listdir(OUT)))